# CQI - Fetch YouTube Transcripts (Colab)

Fetches transcripts for 196 YouTube channels via `youtube-transcript-api` + `yt-dlp`.
Runs from Google's IP (bypasses local IP ban).

**Workflow:**
1. Upload `channels_todo.json` when prompted
2. Run all cells
3. Download `colab_transcripts.json` at the end
4. Copy to local: `/data/projects/creator-quality-index/data/colab_transcripts.json`

In [ ]:
# Cell 1: Install dependencies
!pip install -q youtube-transcript-api yt-dlp

In [ ]:
# Cell 2: Upload channel list
# Upload channels_todo.json (exported from local DB — 196 channels)
import json, os
from google.colab import files

if not os.path.exists('channels_todo.json'):
    print("Upload channels_todo.json (196 channels: 173 new + 23 retry)")
    uploaded = files.upload()
    if 'channels_todo.json' in uploaded:
        CHANNELS = json.loads(uploaded['channels_todo.json'].decode('utf-8'))
    else:
        fname = list(uploaded.keys())[0]
        CHANNELS = json.loads(uploaded[fname].decode('utf-8'))
else:
    with open('channels_todo.json') as f:
        CHANNELS = json.load(f)

print(f"Loaded {len(CHANNELS)} channels to process")

In [ ]:
# Cell 4: Core functions
import subprocess
import time
import json

MAX_TRANSCRIPT_CHARS = 12000
DELAY_OK = 2       # seconds between successful requests
DELAY_FAIL = 3     # seconds after failure
OUTPUT_FILE = 'colab_transcripts.json'


def get_recent_video_id(channel_url):
    """Get most recent video ID from a channel via yt-dlp."""
    try:
        result = subprocess.run(
            ['yt-dlp', '--flat-playlist', '--playlist-end', '3',
             '--print', '%(id)s', f'{channel_url}/videos'],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode == 0 and result.stdout.strip():
            return result.stdout.strip().split('\n')[0]
    except (subprocess.TimeoutExpired, FileNotFoundError):
        pass
    return None


def get_video_title(video_id):
    """Get video title via yt-dlp."""
    try:
        result = subprocess.run(
            ['yt-dlp', '--print', '%(title)s', '--no-download',
             f'https://www.youtube.com/watch?v={video_id}'],
            capture_output=True, text=True, timeout=15
        )
        if result.returncode == 0:
            return result.stdout.strip()
    except (subprocess.TimeoutExpired, FileNotFoundError):
        pass
    return 'Unknown'


def fetch_transcript(video_id, languages):
    """Fetch transcript, return (text, status)."""
    from youtube_transcript_api import YouTubeTranscriptApi
    try:
        ytt = YouTubeTranscriptApi()
        transcript = ytt.fetch(video_id, languages=languages)
        text = ' '.join(snippet.text for snippet in transcript)
        return text, 'ok'
    except Exception as e:
        err = str(e).lower()
        if 'blocking' in err or 'ip' in err:
            return None, 'ip_blocked'
        return None, 'no_transcript'


def check_transcript_quality(text):
    """Detect corrupted Whisper-like transcripts (low unique word ratio)."""
    words = text.lower().split()
    if len(words) < 50:
        return False, 0.0
    unique_ratio = len(set(words)) / len(words)
    return unique_ratio >= 0.20, unique_ratio


def load_results():
    """Load previous results if any."""
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE) as f:
            return json.load(f)
    return []


def save_results(results):
    """Save results incrementally."""
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)


print('Functions loaded OK')

In [ ]:
# Cell 5: Quick IP ban check
from youtube_transcript_api import YouTubeTranscriptApi

try:
    ytt = YouTubeTranscriptApi()
    ytt.fetch('jNQXAC9IVRw')  # First YouTube video ever
    print('IP is CLEAN - ready to fetch!')
except Exception as e:
    print(f'WARNING: IP may be blocked: {e}')
    print('Try: Runtime > Disconnect and delete runtime, then reconnect')

In [ ]:
# Cell 5: MAIN FETCH LOOP
# All 196 channels (new + retry) are in CHANNELS from the uploaded JSON

results = load_results()
done_ids = {r['id'] for r in results}
to_process = [ch for ch in CHANNELS if ch['id'] not in done_ids]

ok_count = sum(1 for r in results if r.get('status') == 'ok')
print(f'Already done: {len(done_ids)} ({ok_count} OK)')
print(f'To process: {len(to_process)}')
print(f'Starting...\n')

consecutive_blocks = 0

for i, ch in enumerate(to_process):
    cid = ch['id']
    lang = ch.get('language') or 'en'
    languages = [lang, 'en'] if lang != 'en' else ['en']
    name = ch['name'][:40]
    
    print(f'[{i+1}/{len(to_process)}] #{cid} {name:40s} ({lang}) ... ', end='', flush=True)
    
    # Get video ID
    video_id = get_recent_video_id(ch['url'])
    if not video_id:
        print('NO VIDEO')
        results.append({'id': cid, 'name': ch['name'], 'language': lang,
                        'tier': ch.get('tier'), 'category': ch.get('primary_category'),
                        'status': 'no_video'})
        save_results(results)
        time.sleep(DELAY_FAIL)
        continue
    
    # Fetch transcript
    text, status = fetch_transcript(video_id, languages)
    
    if status == 'ip_blocked':
        print(f'IP BLOCKED!')
        consecutive_blocks += 1
        if consecutive_blocks >= 3:
            print(f'\n>>> 3 consecutive blocks. Stopping. Resume later.')
            print(f'>>> Progress saved: {len(results)} processed, {ok_count} OK')
            break
        print(f'  Waiting 60s before retry...')
        time.sleep(60)
        # Retry same channel
        text, status = fetch_transcript(video_id, languages)
        if status == 'ip_blocked':
            print(f'  Still blocked. Skipping #{cid}.')
            results.append({'id': cid, 'name': ch['name'], 'language': lang,
                            'tier': ch.get('tier'), 'category': ch.get('primary_category'),
                            'video_id': video_id, 'status': 'ip_blocked'})
            save_results(results)
            continue
    
    consecutive_blocks = 0
    
    if not text:
        print(f'NO TRANSCRIPT (vid={video_id})')
        results.append({'id': cid, 'name': ch['name'], 'language': lang,
                        'tier': ch.get('tier'), 'category': ch.get('primary_category'),
                        'video_id': video_id, 'status': 'no_transcript'})
        save_results(results)
        time.sleep(DELAY_FAIL)
        continue
    
    # Quality check
    is_good, ratio = check_transcript_quality(text)
    quality_status = 'ok' if is_good else 'low_quality'
    
    title = get_video_title(video_id)
    truncated = text[:MAX_TRANSCRIPT_CHARS]
    ok_count += 1
    
    print(f'OK ({len(text)} chars, quality={ratio:.2f}, vid={video_id})')
    results.append({
        'id': cid,
        'name': ch['name'],
        'tier': ch.get('tier'),
        'category': ch.get('primary_category'),
        'language': lang,
        'video_id': video_id,
        'video_title': title,
        'transcript': truncated,
        'transcript_length': len(text),
        'unique_word_ratio': round(ratio, 3),
        'status': quality_status,
    })
    
    save_results(results)
    if ok_count % 10 == 0:
        print(f'  >>> {ok_count} transcripts fetched so far')
    
    time.sleep(DELAY_OK)

# Final summary
save_results(results)
from collections import Counter
stats = Counter(r.get('status') for r in results)
print(f'\n{"="*60}')
print(f'DONE — {len(results)} total processed')
for s, c in stats.most_common():
    print(f'  {s}: {c}')
print(f'Saved to {OUTPUT_FILE}')

In [ ]:
# Cell 7: Download results
from google.colab import files

# Show summary before download
with open(OUTPUT_FILE) as f:
    results = json.load(f)

ok_results = [r for r in results if r.get('status') in ('ok', 'low_quality')]
print(f'Total results: {len(results)}')
print(f'With transcripts: {len(ok_results)}')
print(f'\nDownloading {OUTPUT_FILE}...')

files.download(OUTPUT_FILE)

In [ ]:
# Cell 8 (optional): Quick stats on fetched transcripts
with open(OUTPUT_FILE) as f:
    results = json.load(f)

ok_results = [r for r in results if r.get('status') in ('ok', 'low_quality')]

print(f'Transcripts fetched: {len(ok_results)}')
print(f'\nBy language:')
from collections import Counter
lang_stats = Counter(r['language'] for r in ok_results)
for lang, count in lang_stats.most_common():
    print(f'  {lang}: {count}')

print(f'\nQuality distribution:')
for r in ok_results:
    ratio = r.get('unique_word_ratio', 0)
    if ratio < 0.15:
        print(f'  CORRUPT: #{r["id"]} {r["name"]} (ratio={ratio})')
    elif ratio < 0.20:
        print(f'  BORDERLINE: #{r["id"]} {r["name"]} (ratio={ratio})')

print(f'\nAvg transcript length: {sum(r.get("transcript_length",0) for r in ok_results)//max(len(ok_results),1)} chars')